# Build `silver.nyc_tlc.trips` with PySpark

This notebook builds the **Silver** conformance table `silver.nyc_tlc.trips` from Bronze NYC TLC Yellow and Green raw Parquet files.

## Notebook role in the lakehouse

- **Bronze** keeps raw landed files by source and month
- **Silver** publishes the trusted, conformed, trip-level analytical contract
- **Gold** must read from Silver only

## Business purpose

Publish one canonical trip-level dataset for downstream analytics, quality-controlled exploration, and Gold table construction.

## Target grain

**One row per accepted Bronze trip record after canonicalization, filtering, validation, and exact-duplicate removal.**

## Intended consumers

- Gold transformation notebooks
- Analysts needing validated detailed trip facts
- Engineers validating source quality and schema drift

## Key design choices used in this notebook

- canonical schema across Yellow and Green
- explicit lineage columns
- clear Silver admission filtering logic
- tolerated anomalies kept with quality flags
- exact-duplicate removal based on a deterministic record hash
- Iceberg hidden partitioning by `month(pickup_ts)`
- optional month-scoped Bronze reads to reduce memory pressure

## What this notebook does

1. defines notebook parameters
2. reads Yellow and Green Bronze files
3. validates required source fields
4. maps both sources to one canonical schema
5. applies Silver admission filtering rules
6. removes exact duplicates
7. computes non-blocking quality flags
8. writes the Silver Iceberg table
9. runs post-write validation checks


## Inputs, outputs, and dependencies

### Inputs

- `s3a://bronze/mobility/nyc_tlc/yellow/`
- `s3a://bronze/mobility/nyc_tlc/green/`

### Output

- `silver.nyc_tlc.trips`

### Assumptions

- Spark, Iceberg, Polaris REST catalog, and S3 access are already configured outside the notebook
- sensitive Spark options are intentionally hidden in `spark-defaults`
- Bronze folders are partitioned as `month=YYYY-MM`
- the Bronze landing preserves the raw source columns needed by the official NYC TLC dictionaries

### Important modeling note

This notebook creates a **deterministic record hash** for lineage and exact-duplicate detection. It is **not** presented as a universal business trip identifier.


## Run parameters

This notebook still rebuilds the full `silver.nyc_tlc.trips` table, but it now supports **optional month-scoped Bronze reads**.

Use `PROCESS_MONTHS` when the full Bronze dataset is too large for the available Spark memory and may trigger executor OOM or Kubernetes `OOMKilled` events.

- when `PROCESS_MONTHS = []`, the notebook reads **all available Bronze months**
- when `PROCESS_MONTHS` contains one or more `YYYY-MM` values, the notebook reads **only those Bronze month partitions**

This keeps the notebook simple while making it easier to operate on smaller environments.


## Operational notes

### Why this notebook uses filtering instead of a reject workflow

This notebook is designed to stay easy to understand and easy to operate.

Rows that do not meet the minimum Silver admission rules are simply filtered out before publication, and the notebook reports how many were excluded and why.

### Why `PROCESS_MONTHS` exists

Large Bronze scans can exhaust the available Spark memory and lead to executor OOM or Kubernetes `OOMKilled` events.

Use `PROCESS_MONTHS` to limit the Bronze read scope to selected month partitions on smaller environments. Leave it empty on larger environments when you want to load all available months.

Consider `DISK_ONLY` only as an operational fallback for constrained executors, knowing that without PVC-backed executor volumes the persisted data remains ephemeral.

### What this notebook deliberately does not do

- it does not persist a dedicated reject-table contract
- it does not auto-accept unexpected source schema evolution into the Silver contract
- it does not create a universal business trip ID
- it does not enrich with Taxi Zone metadata in this step

### Known future improvements

- persist a formal quarantine table only if operational auditability becomes necessary
- add referential validation against a curated Taxi Zone dimension
- add automated Iceberg maintenance orchestration for compaction, snapshot expiration, and orphan cleanup
- move shared mapping and validation helpers into reusable Python modules so the notebook stays thinner

## Important runtime note

This notebook sets Spark packages and catalog settings in `SparkSession.builder`. Run it in a fresh kernel so Spark starts with the intended configuration.

The notebook may fail on SeaweedFS with `CreateMultipartUpload`. This is a SeaweedFS issue on the storage write path, not a problem with the notebook logic.



In [1]:
from datetime import datetime, timezone

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F


In [2]:
CATALOG_NAME = "silver"
NAMESPACE = "nyc_tlc"
TABLE_NAME = "trips"
FULL_TABLE_NAME = f"{CATALOG_NAME}.{NAMESPACE}.{TABLE_NAME}"

BRONZE_YELLOW_BASE_PATH = "s3a://bronze/mobility/nyc_tlc/yellow"
BRONZE_GREEN_BASE_PATH = "s3a://bronze/mobility/nyc_tlc/green"
PROCESS_MONTHS = [] # example: ["2025-01", "2025-02"]; empty means all available months

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

TOTAL_AMOUNT_TOLERANCE = 0.05
SUSPICIOUS_DURATION_MINUTES = 24 * 60
SUSPICIOUS_DISTANCE_MILES = 200.0
SUSPICIOUS_PASSENGER_COUNT = 8

VALID_RATE_CODE_IDS = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]
VALID_TRIP_TYPES = [1, 2]
VALID_VENDOR_IDS_BY_SOURCE = {
    "yellow": [1, 2, 6, 7],
    "green": [1, 2, 6],
}

print("FULL_TABLE_NAME:", FULL_TABLE_NAME)
print("PROCESS_MONTHS:", PROCESS_MONTHS if PROCESS_MONTHS else "ALL")
print("RUN_ID:", RUN_ID)

FULL_TABLE_NAME: silver.nyc_tlc.trips
PROCESS_MONTHS: ALL
RUN_ID: 20260326T122902Z


## Spark session and runtime notes

This notebook only sets notebook-local runtime values that are safe to make visible.

Catalog wiring, Iceberg extensions, REST catalog endpoint, credentials, and S3 access are assumed to be injected outside the notebook.


In [3]:
polaris_credential = "my-polaris-spark-etl-app:mySparkAppSecret"
spark = (
    SparkSession.builder
    .appName("PySpark NYC Tripdata - Build Silver 'TRIPS' Table")
    .config(f"spark.sql.catalog.{CATALOG_NAME}.credential", polaris_credential)
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "640m")
    .config("spark.executor.cores", 1)
    .config("spark.kubernetes.executor.request.cores", 1)
    .config("spark.kubernetes.executor.limit.cores", 1)
    .getOrCreate()
)

In [4]:
print("Spark version:", spark.version)
spark.sql("USE silver")
spark.sql("SHOW CATALOGS").show(truncate=False)

Spark version: 3.5.6
+-------------+
|catalog      |
+-------------+
|silver       |
|spark_catalog|
+-------------+



In [5]:
spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME} PURGE")

DataFrame[]

## Read Bronze datasets

This notebook reads either:

- the full Yellow and Green Bronze roots when `PROCESS_MONTHS` is empty, or
- only the selected `month=YYYY-MM` partitions when `PROCESS_MONTHS` is populated

This is especially useful when loading the full Bronze dataset would risk out-of-memory failures.

It also captures file-level lineage using `input_file_name()`.


In [6]:
def build_source_paths(base_path, process_months):
    if process_months:
        return [f"{base_path}/month={month_value}" for month_value in process_months]
    return [base_path]

yellow_paths = build_source_paths(BRONZE_YELLOW_BASE_PATH, PROCESS_MONTHS)
green_paths = build_source_paths(BRONZE_GREEN_BASE_PATH, PROCESS_MONTHS)

print("Yellow input paths:")
for path_value in yellow_paths:
    print(" -", path_value)

print("Green input paths:")
for path_value in green_paths:
    print(" -", path_value)

yellow_raw = spark.read.parquet(*yellow_paths)
green_raw = spark.read.parquet(*green_paths)

print("Yellow schema")
yellow_raw.printSchema()

print("Green schema")
green_raw.printSchema()

print("Yellow rows:", yellow_raw.count())
print("Green rows:", green_raw.count())


Yellow input paths:
 - s3a://bronze/mobility/nyc_tlc/yellow
Green input paths:
 - s3a://bronze/mobility/nyc_tlc/green
Yellow schema
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = 

## Helper functions

These helpers keep the notebook readable and avoid duplicating the canonical mapping logic.


In [7]:
def first_existing_name(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        real_name = lower_map.get(candidate.lower())
        if real_name is not None:
            return real_name
    return None

def required_col(df, candidates, dataset_name, target_type):
    real_name = first_existing_name(df, candidates)
    if real_name is None:
        raise ValueError(
            f"{dataset_name} is missing all candidate columns {candidates}. "
            f"Available columns: {df.columns}"
        )
    return F.col(real_name).cast(target_type)

def optional_col(df, candidates, target_type):
    real_name = first_existing_name(df, candidates)
    if real_name is None:
        return F.lit(None).cast(target_type)
    return F.col(real_name).cast(target_type)

def normalize_store_and_fwd(col_expr):
    return F.when(
        F.upper(F.trim(col_expr)).isin("Y", "N"),
        F.upper(F.trim(col_expr))
    ).otherwise(F.when(col_expr.isNull(), F.lit(None)).otherwise(F.upper(F.trim(col_expr))))

def dataset_vendor_is_valid(source_col, vendor_col):
    return (
        F.when((source_col == F.lit("yellow")) & vendor_col.isin(*VALID_VENDOR_IDS_BY_SOURCE["yellow"]), F.lit(True))
         .when((source_col == F.lit("green")) & vendor_col.isin(*VALID_VENDOR_IDS_BY_SOURCE["green"]), F.lit(True))
         .otherwise(F.lit(False))
    )

def source_month_expr(df):
    source_month_column = optional_col(df, ["month"], "string")
    source_month_from_path = F.regexp_extract(F.input_file_name(), r"month=([0-9]{4}-[0-9]{2})", 1)
    return F.coalesce(source_month_column, source_month_from_path)

def amount_or_zero(col_name):
    return F.coalesce(F.col(col_name), F.lit(0.0))

def add_flag(flag_name, condition):
    return F.when(condition, F.lit(flag_name))

def approx_equal(left_col, right_col, tolerance):
    return F.abs(left_col - right_col) <= F.lit(tolerance)


## Canonical mapping

The canonical Silver schema standardizes the source-specific timestamp columns and preserves source-specific nullable fields where needed.

### Lineage fields included

- `source_dataset`
- `source_month`
- `source_file_path`
- `run_id`
- `ingestion_ts`
- `record_hash`

### Source-specific handling

- `trip_type` exists only for Green and stays nullable for Yellow
- `airport_fee` exists only for Yellow and stays nullable for Green


In [8]:
yellow_canonical = (
    yellow_raw
    .select(
        F.lit("yellow").alias("source_dataset"),
        source_month_expr(yellow_raw).alias("source_month"),
        F.input_file_name().alias("source_file_path"),
        required_col(yellow_raw, ["vendorid", "VendorID"], "yellow", "int").alias("vendor_id"),
        required_col(yellow_raw, ["tpep_pickup_datetime"], "yellow", "timestamp").alias("pickup_ts"),
        required_col(yellow_raw, ["tpep_dropoff_datetime"], "yellow", "timestamp").alias("dropoff_ts"),
        required_col(yellow_raw, ["pulocationid", "PULocationID"], "yellow", "int").alias("pickup_location_id"),
        required_col(yellow_raw, ["dolocationid", "DOLocationID"], "yellow", "int").alias("dropoff_location_id"),
        optional_col(yellow_raw, ["passenger_count"], "int").alias("passenger_count"),
        optional_col(yellow_raw, ["trip_distance"], "double").alias("trip_distance"),
        optional_col(yellow_raw, ["ratecodeid", "RatecodeID"], "int").alias("rate_code_id"),
        normalize_store_and_fwd(optional_col(yellow_raw, ["store_and_fwd_flag"], "string")).alias("store_and_fwd_flag"),
        optional_col(yellow_raw, ["payment_type"], "int").alias("payment_type"),
        F.lit(None).cast("int").alias("trip_type"),
        optional_col(yellow_raw, ["fare_amount"], "double").alias("fare_amount"),
        optional_col(yellow_raw, ["extra"], "double").alias("extra"),
        optional_col(yellow_raw, ["mta_tax"], "double").alias("mta_tax"),
        optional_col(yellow_raw, ["tip_amount"], "double").alias("tip_amount"),
        optional_col(yellow_raw, ["tolls_amount"], "double").alias("tolls_amount"),
        optional_col(yellow_raw, ["improvement_surcharge"], "double").alias("improvement_surcharge"),
        optional_col(yellow_raw, ["total_amount"], "double").alias("total_amount"),
        optional_col(yellow_raw, ["congestion_surcharge"], "double").alias("congestion_surcharge"),
        optional_col(yellow_raw, ["airport_fee"], "double").alias("airport_fee"),
        optional_col(yellow_raw, ["cbd_congestion_fee"], "double").alias("cbd_congestion_fee"),
    )
)

green_canonical = (
    green_raw
    .select(
        F.lit("green").alias("source_dataset"),
        source_month_expr(green_raw).alias("source_month"),
        F.input_file_name().alias("source_file_path"),
        required_col(green_raw, ["vendorid", "VendorID"], "green", "int").alias("vendor_id"),
        required_col(green_raw, ["lpep_pickup_datetime"], "green", "timestamp").alias("pickup_ts"),
        required_col(green_raw, ["lpep_dropoff_datetime"], "green", "timestamp").alias("dropoff_ts"),
        required_col(green_raw, ["pulocationid", "PULocationID"], "green", "int").alias("pickup_location_id"),
        required_col(green_raw, ["dolocationid", "DOLocationID"], "green", "int").alias("dropoff_location_id"),
        optional_col(green_raw, ["passenger_count"], "int").alias("passenger_count"),
        optional_col(green_raw, ["trip_distance"], "double").alias("trip_distance"),
        optional_col(green_raw, ["ratecodeid", "RatecodeID"], "int").alias("rate_code_id"),
        normalize_store_and_fwd(optional_col(green_raw, ["store_and_fwd_flag"], "string")).alias("store_and_fwd_flag"),
        optional_col(green_raw, ["payment_type"], "int").alias("payment_type"),
        optional_col(green_raw, ["trip_type"], "int").alias("trip_type"),
        optional_col(green_raw, ["fare_amount"], "double").alias("fare_amount"),
        optional_col(green_raw, ["extra"], "double").alias("extra"),
        optional_col(green_raw, ["mta_tax"], "double").alias("mta_tax"),
        optional_col(green_raw, ["tip_amount"], "double").alias("tip_amount"),
        optional_col(green_raw, ["tolls_amount"], "double").alias("tolls_amount"),
        optional_col(green_raw, ["improvement_surcharge"], "double").alias("improvement_surcharge"),
        optional_col(green_raw, ["total_amount"], "double").alias("total_amount"),
        optional_col(green_raw, ["congestion_surcharge"], "double").alias("congestion_surcharge"),
        F.lit(None).cast("double").alias("airport_fee"),
        optional_col(green_raw, ["cbd_congestion_fee"], "double").alias("cbd_congestion_fee"),
    )
)

canonical_union = yellow_canonical.unionByName(green_canonical)

canonical_union.printSchema()
canonical_union.show(5, truncate=False)


root
 |-- source_dataset: string (nullable = false)
 |-- source_month: string (nullable = false)
 |-- source_file_path: string (nullable = false)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable

## Silver admission filtering logic

This notebook uses **filtering logic** rather than a separate reject workflow.

A record is **included in Silver** only if it satisfies the minimum conditions required for a trustworthy trip-level analytical contract.

### Records are filtered out before Silver publish when

- `pickup_ts` is missing
- `dropoff_ts` is missing
- `pickup_location_id` is missing
- `dropoff_location_id` is missing
- trip duration is negative
- trip distance is negative
- passenger count is negative
- `source_month` is missing
- `source_month` is inconsistent with `pickup_ts`

These are treated as **minimum inclusion rules** for Silver, not as a separate reject-table design.

### Records kept in Silver with quality flags

Rows that remain analytically usable but look suspicious are kept in Silver and marked with `dq_flags`.

This makes the Silver dataset both trustworthy and practical for downstream use.


In [9]:
canonical_enriched = (
    canonical_union
    .withColumn("pickup_date", F.to_date("pickup_ts"))
    .withColumn("pickup_month", F.date_format("pickup_ts", "yyyy-MM"))
    .withColumn(
        "trip_duration_minutes",
        (F.col("dropoff_ts").cast("long") - F.col("pickup_ts").cast("long")) / F.lit(60.0)
    )
    .withColumn("run_id", F.lit(RUN_ID))
    .withColumn("ingestion_ts", F.current_timestamp())
)

record_hash_struct = F.struct(
    "source_dataset",
    "source_month",
    "vendor_id",
    "pickup_ts",
    "dropoff_ts",
    "pickup_location_id",
    "dropoff_location_id",
    "passenger_count",
    "trip_distance",
    "rate_code_id",
    "store_and_fwd_flag",
    "payment_type",
    "trip_type",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
)

quality_base = canonical_enriched.withColumn(
    "record_hash",
    F.sha2(F.to_json(record_hash_struct), 256)
)

quality_base = (
    quality_base
    .withColumn("filter_missing_pickup_ts", F.col("pickup_ts").isNull())
    .withColumn("filter_missing_dropoff_ts", F.col("dropoff_ts").isNull())
    .withColumn("filter_missing_pickup_location_id", F.col("pickup_location_id").isNull())
    .withColumn("filter_missing_dropoff_location_id", F.col("dropoff_location_id").isNull())
    .withColumn(
        "filter_missing_source_month",
        F.col("source_month").isNull() | (F.trim(F.col("source_month")) == "")
    )
    .withColumn("filter_negative_trip_duration", F.col("trip_duration_minutes") < 0)
    .withColumn("filter_negative_trip_distance", F.col("trip_distance") < 0)
    .withColumn("filter_negative_passenger_count", F.col("passenger_count") < 0)
    .withColumn(
        "filter_source_month_mismatch",
        F.col("pickup_ts").isNotNull()
        & F.col("source_month").isNotNull()
        & (F.col("pickup_month") != F.col("source_month"))
    )
)

quality_base = quality_base.withColumn(
    "filter_out_reasons",
    F.filter(
        F.array(
            F.when(F.col("filter_missing_pickup_ts"), F.lit("missing_pickup_ts")),
            F.when(F.col("filter_missing_dropoff_ts"), F.lit("missing_dropoff_ts")),
            F.when(F.col("filter_missing_pickup_location_id"), F.lit("missing_pickup_location_id")),
            F.when(F.col("filter_missing_dropoff_location_id"), F.lit("missing_dropoff_location_id")),
            F.when(F.col("filter_missing_source_month"), F.lit("missing_source_month")),
            F.when(F.col("filter_negative_trip_duration"), F.lit("negative_trip_duration")),
            F.when(F.col("filter_negative_trip_distance"), F.lit("negative_trip_distance")),
            F.when(F.col("filter_negative_passenger_count"), F.lit("negative_passenger_count")),
            F.when(F.col("filter_source_month_mismatch"), F.lit("source_month_mismatch")),
        ),
        lambda x: x.isNotNull()
    )
)

quality_base = quality_base.withColumn(
    "is_filtered_out",
    F.size(F.col("filter_out_reasons")) > 0
)

filtered_out_records = quality_base.where(F.col("is_filtered_out")).cache()
filter_pass_records = quality_base.where(~F.col("is_filtered_out")).cache()

## Exact-duplicate handling

There is no trusted universal trip identifier in the raw source schema, so this notebook only removes **exact duplicates** using a deterministic hash over the canonical business columns.

Near-duplicates are not merged.


In [10]:
dedup_window = Window.partitionBy("record_hash").orderBy(
    F.col("source_file_path").asc(),
    F.col("source_dataset").asc()
)

ranked_records = filter_pass_records.withColumn("duplicate_rank", F.row_number().over(dedup_window))

exact_duplicates = ranked_records.where(F.col("duplicate_rank") > 1).drop("duplicate_rank").cache()

silver_base = ranked_records.where(F.col("duplicate_rank") == 1).drop("duplicate_rank").cache()


## Non-blocking data-quality flags

Rows that pass the Silver admission filters may still contain suspicious values.

These rows are retained in Silver and marked with `dq_flags` so downstream users can decide how strictly to consume them.

This section does **not** exclude records from Silver.


In [11]:
expected_total_amount = (
    amount_or_zero("fare_amount")
    + amount_or_zero("extra")
    + amount_or_zero("mta_tax")
    + amount_or_zero("tolls_amount")
    + amount_or_zero("improvement_surcharge")
    + amount_or_zero("congestion_surcharge")
    + amount_or_zero("airport_fee")
    + amount_or_zero("cbd_congestion_fee")
    + F.when(F.col("payment_type") == F.lit(2), F.lit(0.0)).otherwise(amount_or_zero("tip_amount"))
)

negative_monetary_component = (
    (F.col("fare_amount") < 0)
    | (F.col("extra") < 0)
    | (F.col("mta_tax") < 0)
    | (F.col("tip_amount") < 0)
    | (F.col("tolls_amount") < 0)
    | (F.col("improvement_surcharge") < 0)
    | (F.col("total_amount") < 0)
    | (F.col("congestion_surcharge") < 0)
    | (F.col("airport_fee") < 0)
    | (F.col("cbd_congestion_fee") < 0)
)

silver_trips = (
    silver_base
    .withColumn(
        "dq_flags",
        F.filter(
            F.array(
                add_flag("unknown_vendor_id", ~dataset_vendor_is_valid(F.col("source_dataset"), F.col("vendor_id"))),
                add_flag("unknown_rate_code_id", F.col("rate_code_id").isNotNull() & ~F.col("rate_code_id").isin(*VALID_RATE_CODE_IDS)),
                add_flag("unknown_payment_type", F.col("payment_type").isNotNull() & ~F.col("payment_type").isin(*VALID_PAYMENT_TYPES)),
                add_flag(
                    "unknown_trip_type",
                    (F.col("source_dataset") == F.lit("green"))
                    & F.col("trip_type").isNotNull()
                    & ~F.col("trip_type").isin(*VALID_TRIP_TYPES)
                ),
                add_flag(
                    "invalid_store_and_fwd_flag",
                    F.col("store_and_fwd_flag").isNotNull() & ~F.col("store_and_fwd_flag").isin("Y", "N")
                ),
                add_flag("suspicious_trip_duration", F.col("trip_duration_minutes") > F.lit(SUSPICIOUS_DURATION_MINUTES)),
                add_flag("suspicious_trip_distance", F.col("trip_distance") > F.lit(SUSPICIOUS_DISTANCE_MILES)),
                add_flag("suspicious_passenger_count", F.col("passenger_count") > F.lit(SUSPICIOUS_PASSENGER_COUNT)),
                add_flag("negative_monetary_component", negative_monetary_component),
                add_flag(
                    "total_amount_mismatch_heuristic",
                    F.col("total_amount").isNotNull()
                    & F.col("payment_type").isin(1, 2)
                    & ~approx_equal(F.col("total_amount"), expected_total_amount, TOTAL_AMOUNT_TOLERANCE)
                ),
            ),
            lambda x: x.isNotNull()
        )
    )
    .withColumn("dq_flag_count", F.size("dq_flags"))
    .withColumn(
        "quality_status",
        F.when(F.col("dq_flag_count") > 0, F.lit("accepted_with_flags")).otherwise(F.lit("accepted_clean"))
    )
    .cache()
)

silver_trips.printSchema()
silver_trips.show(5, truncate=False)

root
 |-- source_dataset: string (nullable = false)
 |-- source_month: string (nullable = false)
 |-- source_file_path: string (nullable = false)
 |-- vendor_id: integer (nullable = true)
 |-- pickup_ts: timestamp (nullable = true)
 |-- dropoff_ts: timestamp (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- rate_code_id: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- trip_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable

## Validation report

This section gives an operator-friendly summary before the table is written.

It distinguishes:

- raw input rows
- rows filtered out by Silver admission rules
- exact duplicates removed after filtering
- rows accepted into Silver
- accepted rows that still carry quality flags


In [12]:
ENABLE_VALIDATION_REPORT = False
# This step may take a lot of time, enable it according to your environment
if ENABLE_VALIDATION_REPORT:
    yellow_count = yellow_raw.count()
    green_count = green_raw.count()
    raw_total = yellow_count + green_count
    canonical_total = canonical_union.count()
    filtered_out_total = filtered_out_records.count()
    exact_duplicate_total = exact_duplicates.count()
    silver_total = silver_trips.count()
    
    print("Yellow raw rows:", yellow_count)
    print("Green raw rows:", green_count)
    print("Raw total rows:", raw_total)
    print("Canonical mapped rows:", canonical_total)
    print("Filtered out rows:", filtered_out_total)
    print("Exact duplicates removed:", exact_duplicate_total)
    print("Silver accepted rows:", silver_total)
    
    print("\nAccepted rows by source and month")
    silver_trips.groupBy("source_dataset", "source_month", "quality_status").count().orderBy(
        "source_dataset", "source_month", "quality_status"
    ).show(10, truncate=False)
    
    print("\nFiltered-out rows by reason")
    filtered_out_records.select(F.explode("filter_out_reasons").alias("filter_out_reason")).groupBy("filter_out_reason").count().orderBy(F.desc("count")).show(100, truncate=False)
    
    print("\nExact duplicate sample")
    exact_duplicates.select(
        "source_dataset",
        "source_month",
        "source_file_path",
        "pickup_ts",
        "dropoff_ts",
        "pickup_location_id",
        "dropoff_location_id",
        "record_hash",
    ).show(10, truncate=False)
    
    print("\nAccepted rows with quality flags")
    silver_trips.where(F.col("dq_flag_count") > 0).select(
        "source_dataset",
        "source_month",
        "pickup_ts",
        "dropoff_ts",
        "pickup_location_id",
        "dropoff_location_id",
        "trip_distance",
        "total_amount",
        "dq_flags",
    ).show(10, truncate=False)
    
    print("\nFiltered-out sample")
    filtered_out_records.select(
        "source_dataset",
        "source_month",
        "source_file_path",
        "pickup_ts",
        "dropoff_ts",
        "pickup_location_id",
        "dropoff_location_id",
        "filter_out_reasons",
    ).show(10, truncate=False)
else:
    print("Validation report disabled.")

Yellow raw rows: 11198026
Green raw rows: 146486
Raw total rows: 11344512
Canonical mapped rows: 11344512
Filtered out rows: 975
Exact duplicates removed: 0
Silver accepted rows: 11343537

Accepted rows by source and month
+--------------+------------+-------------------+-------+
|source_dataset|source_month|quality_status     |count  |
+--------------+------------+-------------------+-------+
|green         |2025-01     |accepted_clean     |42120  |
|green         |2025-01     |accepted_with_flags|6163   |
|green         |2025-02     |accepted_clean     |40481  |
|green         |2025-02     |accepted_with_flags|5911   |
|green         |2025-03     |accepted_clean     |44745  |
|green         |2025-03     |accepted_with_flags|6475   |
|yellow        |2025-01     |accepted_clean     |2731838|
|yellow        |2025-01     |accepted_with_flags|743242 |
|yellow        |2025-02     |accepted_clean     |2811646|
|yellow        |2025-02     |accepted_with_flags|765773 |
+--------------+-------

## Publish the Silver Iceberg table

### Write strategy

This notebook always performs a **full refresh** using `CREATE OR REPLACE TABLE`.

### Partition strategy

The Silver table uses hidden partitioning by `month(pickup_ts)`.

This aligns well with:

- Bronze source organization by month
- the operational need to query and validate data by pickup month
- downstream aggregate construction in Gold

### Schema evolution posture

This notebook is intentionally conservative:

- known optional source-specific fields are supported
- unexpected contract changes should fail visibly and be reviewed before being accepted into Silver


In [13]:
spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG_NAME}.{NAMESPACE}")

silver_trips_view = "silver_trips_stage_for_write"
silver_trips.createOrReplaceTempView(silver_trips_view)

spark.sql(f'''
CREATE OR REPLACE TABLE {FULL_TABLE_NAME}
USING iceberg
PARTITIONED BY (month(pickup_ts))
TBLPROPERTIES (
  'format-version' = '2',
  'write.distribution-mode' = 'hash',
  'comment' = 'Canonical conformed NYC TLC yellow and green trip records in the Silver layer'
)
AS
SELECT
  source_dataset,
  source_month,
  source_file_path,
  run_id,
  ingestion_ts,
  record_hash,
  vendor_id,
  pickup_ts,
  dropoff_ts,
  pickup_date,
  pickup_month,
  trip_duration_minutes,
  pickup_location_id,
  dropoff_location_id,
  passenger_count,
  trip_distance,
  rate_code_id,
  store_and_fwd_flag,
  payment_type,
  trip_type,
  fare_amount,
  extra,
  mta_tax,
  tip_amount,
  tolls_amount,
  improvement_surcharge,
  total_amount,
  congestion_surcharge,
  airport_fee,
  cbd_congestion_fee,
  dq_flags,
  dq_flag_count,
  quality_status
FROM {silver_trips_view}
''')

print(f"Completed full refresh of {FULL_TABLE_NAME}")


Py4JJavaError: An error occurred while calling o90.sql.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 1 in stage 57.0 failed 4 times, most recent failure: Lost task 1.3 in stage 57.0 (TID 1468) (10.244.0.150 executor 2): software.amazon.awssdk.services.s3.model.S3Exception: Access Denied. (Service: S3, Status Code: 403, Request ID: 1774528528901233888) (SDK Attempt Count: 1)
	at software.amazon.awssdk.services.s3.model.S3Exception$BuilderImpl.build(S3Exception.java:113)
	at software.amazon.awssdk.services.s3.model.S3Exception$BuilderImpl.build(S3Exception.java:61)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.utils.RetryableStageHelper.retryPolicyDisallowedRetryException(RetryableStageHelper.java:168)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.RetryableStage.execute(RetryableStage.java:73)
	at software.amazon.awssdk.core.internal.http.pipeline.stages.RetryableStage.execute(RetryableStage.java:36)
	at software.amazon.awssdk.

## Post-write checks

These checks confirm that the published Silver table is readable and consistent with the staged accepted dataset.


In [ ]:
published_count = spark.table(FULL_TABLE_NAME).count()

print("Expected published rows:", silver_total)
print("Actual published rows:", published_count)

if published_count != silver_total:
    raise AssertionError(
        f"Published row count mismatch: expected {silver_total}, got {published_count}"
    )

spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{NAMESPACE}").show(truncate=False)
spark.sql(f"DESCRIBE TABLE {FULL_TABLE_NAME}").show(200, truncate=False)

spark.sql(f'''
SELECT
  source_dataset,
  source_month,
  quality_status,
  COUNT(*) AS row_count
FROM {FULL_TABLE_NAME}
GROUP BY source_dataset, source_month, quality_status
ORDER BY source_dataset, source_month, quality_status
''').show(500, truncate=False)

spark.sql(f'''
SELECT
  COUNT(DISTINCT record_hash) AS distinct_record_hashes,
  COUNT(*) AS total_rows
FROM {FULL_TABLE_NAME}
''').show(truncate=False)

spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 20").show(truncate=False)

try:
    spark.sql(f"SELECT * FROM {FULL_TABLE_NAME}.snapshots").show(truncate=False)
except Exception as exc:
    print("Snapshot metadata query skipped:", exc)
